<a href="https://colab.research.google.com/github/Jobmrtall/MSC-thesis-syntehic-data-/blob/main/Last_close_to_real_data_generating_code_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import re
import uuid
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd

# =============================================================================
# FINAL integrated synthetic ICU generator (2020 start)
# - 6039 distinct patients (distributed by your hospital totals)
# - 1–3 ICU stays per patient (readmissions), flag readmission within 6 months
# - Short stays (1–3 days) included for transfers/early step-down
# - Sites + Black Lion ICU sub-units with machine/bed ratios
# - YEKATIT has mv_data_available=0 (no MV charting)
# - Vitals include resp_device/mode/fio2_equiv_pct (never missing)
# - IMV modes: AC/SIMV/PSV/CPAP + DIRECT_OXYGEN SBT episodes
# - SBT failures sometimes revert to CPAP/AC
# - Accidental extubation: no explicit timestamp column; appears as pattern in
#   resp_device/mode + physician note mention inferred by heuristics
# - Medication orders + MAR admin sheets + note integration (physician includes
#   current + changed; nursing mentions only key meds)
# - Missingness/noise rules: never delete rows; keep required fields non-missing
# =============================================================================

# ============================
# Seed / cohort size
# ============================
SEED = 7

# Distinct patients == sum of your site totals
N_PATIENTS = 6039
MAX_STAYS_PER_PATIENT = 3  # at most 3 ICU stays per patient

# Timeline
START_DATE = pd.Timestamp("2020-01-01 00:00:00")
# Use 2-year horizon to allow readmissions within 6 months for late-year admits
END_DATE = START_DATE + pd.Timedelta(days=365 * 2)

# ============================
# Site distribution (exact totals for PATIENTS)
# ============================
SITE_COUNTS = {
    "ZEWDITU": 1114,
    "BLACK_LION": 1864,
    "RAS_DESTA": 911,
    "MENILIK": 1078,
    "YEKATIT": 1072,
}
assert sum(SITE_COUNTS.values()) == N_PATIENTS

# Black Lion ICU sub-units and capacity
BL_UNITS = {
    "BL_SURGICAL": {"beds": 6, "machines": 6},
    "BL_MEDICAL": {"beds": 6, "machines": 6},
    "BL_GENERAL": {"beds": 8, "machines": 3},
    "BL_CARDIAC": {"beds": 8, "machines": 8},
}
BL_UNIT_PROBS = [
    ("BL_SURGICAL", 0.25),
    ("BL_MEDICAL", 0.25),
    ("BL_GENERAL", 0.25),
    ("BL_CARDIAC", 0.25),
]

SITE_CAPACITY = {
    "ZEWDITU": {"beds": 12, "machines": 8},
    "BLACK_LION": {"beds": None, "machines": None},  # modeled via BL_UNITS
    "RAS_DESTA": {"beds": 5, "machines": 4},
    "MENILIK": {"beds": 8, "machines": 7},
    "YEKATIT": {"beds": None, "machines": None},     # data availability issue, not capacity
}

# ============================
# LOS distribution (days)
# ============================
LOS_MIX = [
    (0.10, 1, 3),   # short stays 1–3 days (transfer/stepdown)
    (0.65, 5, 14),
    (0.20, 15, 45),
    (0.05, 46, 90),
]

# ============================
# Scenario distribution
# ============================
SCENARIOS = [
    ("success_o2", 0.34),
    ("extub_niv_short", 0.18),
    ("fail_reintub_72h", 0.18),
    ("fail_niv_7d_then_imv", 0.08),
    ("prolonged_imv_or_trach", 0.08),
    ("never_intubated_o2", 0.10),
    ("never_intubated_room_air", 0.04),
]

# Accidental extubation (intubated stays only)
P_ACC_EXTUB = 0.04  # ~3–5% target, set to 4%

# ============================
# Notes noise parameters
# ============================
P_OMIT_EVENT = 0.20
P_QUALITATIVE_ONLY = 0.25
P_COPY_FORWARD_NOTE = 0.15
P_TIME_AMBIGUOUS = 0.30
P_IO_NOTE_MISMATCH = 0.12

# ============================
# Column constraints
# ============================
_VITALS_NEVER_MISSING = {"resp_device", "mode", "fio2_equiv_pct"}
_VITALS_NUM_COLS = [
    "hr_bpm", "rr_bpm", "spo2_pct", "map_mmhg", "sbp_mmhg", "dbp_mmhg",
    "temp_c", "glucose_mgdl", "o2_flow_lpm"
]
_MV_VENT_COLS = ["peep", "pip_cmH2O", "tv_set_ml", "rr_set"]
_TIMESTAMP_RE = re.compile(r"\b20\d{2}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}:\d{2})?\b")

# =============================================================================
# Helpers
# =============================================================================
def uid(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def weighted_choice(items):
    total = sum(w for _, w in items)
    r = random.random() * total
    cum = 0.0
    for v, w in items:
        cum += w
        if r <= cum:
            return v
    return items[-1][0]

def pick_from_mixture(mix):
    r = random.random()
    cum = 0.0
    for prob, a, b in mix:
        cum += prob
        if r <= cum:
            return random.randint(a, b)
    return random.randint(mix[-1][1], mix[-1][2])

def hour_range(t0, t1):
    return pd.date_range(t0, t1, freq="h", inclusive="left")

def q3d_range(t0, t1):
    return pd.date_range(t0, t1, freq="3D", inclusive="left")

def rand_time_between(start, end):
    if end <= start:
        return start
    delta_h = int((end - start).total_seconds() / 3600)
    return start + pd.Timedelta(hours=random.randint(0, max(1, delta_h - 1)))

def _ensure_sorted(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    return df.sort_values(by=by, kind="stable").reset_index(drop=True)

def _poisson_capped(lam: float, cap: int) -> int:
    return int(min(np.random.poisson(lam), cap))

def _pick_block_len_hours() -> int:
    r = random.random()
    if r < 0.70:
        return random.randint(1, 3)
    if r < 0.95:
        return random.randint(4, 12)
    return random.randint(24, 48)

def _pick_start_index_weighted(times: pd.Series) -> int:
    n = len(times)
    if n == 0:
        return 0
    admit = times.iloc[0]
    hours_from_admit = ((times - admit) / pd.Timedelta(hours=1)).astype(float)
    hour_of_day = times.dt.hour
    w = np.ones(n, dtype=float)
    w += np.where((hour_of_day >= 0) & (hour_of_day <= 6), 1.0, 0.0)
    w += np.where(hours_from_admit <= 6.0, 1.0, 0.0)
    w = w / w.sum()
    return int(np.random.choice(np.arange(n), p=w))

def _logistic(x):
    return 1.0 / (1.0 + np.exp(-x))

def _additive_measurement_noise(series: pd.Series, sd: float) -> pd.Series:
    s = series.astype(float).copy()
    mask = s.notna()
    s.loc[mask] = s.loc[mask] + np.random.normal(0.0, sd, size=int(mask.sum()))
    return s

def _clip_int_series(series: pd.Series, lo: int, hi: int) -> pd.Series:
    s = series.copy()
    mask = s.notna()
    s.loc[mask] = s.loc[mask].clip(lower=lo, upper=hi).round().astype("Int64")
    return s

def _clip_float_series(series: pd.Series, lo: float, hi: float, decimals: int = 1) -> pd.Series:
    s = series.copy().astype(float)
    mask = s.notna()
    s.loc[mask] = s.loc[mask].clip(lower=lo, upper=hi).round(decimals)
    return s

def _black_lion_unit_choice() -> str:
    return weighted_choice(BL_UNIT_PROBS)

def _assigned_machine_bed(site: str, icu_unit: str | None) -> int:
    if site == "BLACK_LION" and icu_unit is not None:
        cap = BL_UNITS[icu_unit]
        beds = cap["beds"]
        machines = cap["machines"]
    else:
        cap = SITE_CAPACITY.get(site, {})
        beds = cap.get("beds")
        machines = cap.get("machines")
    if beds is None or machines is None:
        p = 0.85
    else:
        p = clamp(machines / beds, 0.0, 1.0)
    return int(random.random() < p)

# =============================================================================
# Truth
# =============================================================================
@dataclass
class StayTruth:
    scenario: str
    admit: pd.Timestamp
    discharge: pd.Timestamp
    extub_time: pd.Timestamp | None
    reintub_time: pd.Timestamp | None
    death_time: pd.Timestamp | None

# =============================================================================
# Patient + stay generation
# =============================================================================
def generate_patient(patient_id: str, site: str):
    sex = weighted_choice([("M", 0.55), ("F", 0.45)])
    age = int(clamp(np.random.normal(62, 14), 18, 95))
    height = int(clamp(np.random.normal(170, 10), 145, 200))
    weight = int(clamp(np.random.normal(80, 18), 40, 160))
    bmi = round(weight / ((height / 100) ** 2), 1)
    smoking = weighted_choice([("Never", 0.45), ("Former", 0.35), ("Current", 0.15), ("Unknown", 0.05)])
    return {
        "patient_id": patient_id,
        "site": site,  # patient "home" site
        "sex": sex,
        "age_years": age,
        "height_cm": height,
        "weight_kg": weight,
        "bmi": bmi,
        "smoking_status": smoking,
    }

def generate_stay_times(base_start: pd.Timestamp) -> tuple[pd.Timestamp, pd.Timestamp]:
    admit = base_start + pd.Timedelta(hours=random.randint(0, 23))
    los_days = pick_from_mixture(LOS_MIX)
    discharge = admit + pd.Timedelta(days=los_days) + pd.Timedelta(hours=random.randint(0, 12))
    return admit, discharge

def generate_truth_course(admit, discharge):
    scenario = weighted_choice([(s, p) for s, p in SCENARIOS])
    hours = list(hour_range(admit, discharge))

    extub_time = None
    reintub_time = None
    death_time = None

    # Mild death probability; allow even in short stays
    p_death = 0.06 + 0.001 * (len(hours) / 24)
    if len(hours) >= 12 and random.random() < min(p_death, 0.25):
        t = admit + pd.Timedelta(hours=random.randint(12, max(13, len(hours) - 1)))
        death_time = t
        discharge = max(discharge, death_time + pd.Timedelta(hours=1))
        hours = list(hour_range(admit, discharge))

    # Planned extubation only in intubated scenarios; prolonged may not extubate
    if scenario in ("success_o2", "extub_niv_short", "fail_reintub_72h", "fail_niv_7d_then_imv", "prolonged_imv_or_trach"):
        if scenario != "prolonged_imv_or_trach" or random.random() < 0.35:
            # for very short stays, extub can happen earlier
            extub_hour = random.randint(6, max(8, int(0.5 * len(hours))))
            extub_time = hours[extub_hour]

    if scenario == "fail_reintub_72h" and extub_time is not None:
        reintub_time = extub_time + pd.Timedelta(hours=random.randint(6, 72))
        if reintub_time >= discharge:
            reintub_time = discharge - pd.Timedelta(hours=2)

    if scenario == "fail_niv_7d_then_imv" and extub_time is not None:
        delta_h = int(clamp(np.random.normal(168, 18), 120, 240))
        reintub_time = extub_time + pd.Timedelta(hours=delta_h)
        if reintub_time >= discharge:
            reintub_time = discharge - pd.Timedelta(hours=2)

    return StayTruth(scenario, admit, discharge, extub_time, reintub_time, death_time)

def generate_static_stay_vars(truth: StayTruth):
    apache = int(clamp(np.random.normal(18, 7), 5, 45))
    sofa_day1 = int(clamp(np.random.normal(7, 4), 0, 20))
    aki_any = int(random.random() < 0.25)
    shock_any = int(random.random() < 0.18)
    vasopressor_any = int(shock_any == 1 and random.random() < 0.75)
    vap_any = int(random.random() < 0.10)
    steroid_any = int(random.random() < 0.22)

    disposition = "Alive"
    if truth.death_time is not None and truth.death_time < truth.discharge:
        disposition = "Death"

    primary_dx = weighted_choice([
        ("Pneumonia", 0.25),
        ("Sepsis", 0.20),
        ("COPD Exacerbation", 0.15),
        ("ARDS", 0.10),
        ("CHF Exacerbation", 0.10),
        ("Post-op", 0.10),
        ("Other", 0.10),
    ])

    # Explicit tags requested
    resp_failure_related = int(
        truth.scenario not in ["never_intubated_room_air"] or
        primary_dx in ["Pneumonia", "COPD Exacerbation", "ARDS"]
    )
    sepsis_related = int(primary_dx == "Sepsis" or shock_any == 1)
    cardiac_related = int(primary_dx == "CHF Exacerbation")
    neuromuscular_related = 0

    # Transfer/discharge destination (simple)
    discharge_destination = weighted_choice([
        ("WARD", 0.70),
        ("OTHER_HOSPITAL", 0.08),
        ("ANOTHER_ICU", 0.07),
        ("HOME", 0.10),
        ("UNKNOWN", 0.05),
    ])
    if disposition == "Death":
        discharge_destination = "MORGUE"

    return {
        "admit_time": truth.admit,
        "discharge_time": truth.discharge,
        "discharge_disposition": disposition,
        "discharge_destination": discharge_destination,
        "death_time": truth.death_time,
        "primary_diagnosis": primary_dx,
        "apache_ii": apache,
        "sofa_day1": sofa_day1,
        "aki_any": aki_any,
        "shock_any": shock_any,
        "vasopressor_any": vasopressor_any,
        "vap_any": vap_any,
        "steroid_any": steroid_any,
        "resp_failure_related": resp_failure_related,
        "sepsis_related": sepsis_related,
        "cardiac_related": cardiac_related,
        "neuromuscular_related": neuromuscular_related,
    }

# =============================================================================
# Respiratory demand, devices, SBT
# =============================================================================
def o2_flow_to_fio2_equiv_pct(flow_lpm: int) -> int:
    fio2 = 21 + int(round(4 * flow_lpm))
    return int(clamp(fio2, 21, 90))

def _generate_oxygen_demand_trajectory(times: pd.DatetimeIndex, trend: str) -> np.ndarray:
    n = len(times)
    if n == 0:
        return np.array([])
    base = np.random.beta(2, 3)
    noise = np.random.normal(0, 0.04, size=n).cumsum()
    if trend == "improve":
        drift = np.linspace(0.15, -0.25, n)
    elif trend == "worsen":
        drift = np.linspace(-0.05, 0.25, n)
    else:
        drift = np.linspace(0.10, -0.10, n)
    demand = base + drift + noise
    return np.clip(demand, 0.0, 1.2)

def _demand_to_device_flow_directional(demand: float, prev_device: str | None, delta: float) -> tuple[str, float | None]:
    if demand <= 0.10:
        base_dev, base_flow = "ROOM_AIR", None
    elif demand <= 0.40:
        flow = int(clamp(round(1 + demand / 0.40 * 4), 1, 5))
        base_dev, base_flow = "NASAL_CANNULA", flow
    elif demand <= 0.70:
        flow = int(clamp(round(6 + (demand - 0.40) / 0.30 * 4), 6, 10))
        base_dev, base_flow = "FACE_MASK", flow
    else:
        flow = int(clamp(round(11 + (demand - 0.70) / 0.30 * 9), 11, 20))
        base_dev, base_flow = "FMWR", flow

    if prev_device is None:
        return base_dev, base_flow

    # Step-up uses FMWR more strongly
    if delta > 0.05:
        if prev_device == "NASAL_CANNULA" and base_dev in ("NASAL_CANNULA", "FACE_MASK"):
            return "FACE_MASK", int(clamp((base_flow or 8), 6, 10))
        if prev_device == "FACE_MASK" and base_dev == "FMWR":
            return "FMWR", int(clamp((base_flow or 12), 11, 20))
        if prev_device == "ROOM_AIR":
            return "NASAL_CANNULA", 2

    # Step-down can skip FMWR
    if delta < -0.03:
        if prev_device == "FMWR":
            if demand <= 0.5:
                return "NASAL_CANNULA", 4
            return "FACE_MASK", 8
        if prev_device == "FACE_MASK" and demand <= 0.3:
            return "NASAL_CANNULA", 3
        if prev_device == "NASAL_CANNULA" and demand <= 0.1:
            return "ROOM_AIR", None

    return base_dev, base_flow

def _plan_sbt_episodes(times: pd.DatetimeIndex, extub_time: pd.Timestamp | None) -> list[list[pd.Timestamp]]:
    episodes = []
    if extub_time is None:
        return episodes
    if random.random() > 0.60:
        return episodes

    start_window = extub_time - pd.Timedelta(hours=48)
    cand = [t for t in times if start_window <= t < extub_time]
    if not cand:
        return episodes

    by_day: dict[pd.Timestamp, list[pd.Timestamp]] = {}
    for t in cand:
        d = t.floor("D")
        by_day.setdefault(d, []).append(t)

    for _, ts in by_day.items():
        if random.random() < 0.60:
            t0 = random.choice(ts)
            ep = [t0]
            t1 = t0 + pd.Timedelta(hours=1)
            if t1 in set(times):
                ep.append(t1)
            episodes.append(ep)

    return episodes

# =============================================================================
# Vitals generator (includes accidental extubation as pattern; no explicit column)
# =============================================================================
def generate_vitals_hourly_with_resp_delivery(icu_stay_id, truth: StayTruth, shock_any=0):
    times = pd.DatetimeIndex(hour_range(truth.admit, truth.discharge))

    if truth.scenario == "never_intubated_room_air":
        demand = _generate_oxygen_demand_trajectory(times, trend="improve")
        demand = np.clip(demand - 0.25, 0.0, 0.8)
    elif truth.scenario == "never_intubated_o2":
        demand = _generate_oxygen_demand_trajectory(times, trend="mixed")
    else:
        demand = _generate_oxygen_demand_trajectory(times, trend="improve")

    sbt_episodes = _plan_sbt_episodes(times, truth.extub_time)
    sbt_hours = {t for ep in sbt_episodes for t in ep}

    # Accidental extubation: choose internal time + outcome; DO NOT store as column
    accidental_time = None
    accidental_path_deteriorate = False
    if truth.scenario not in ("never_intubated_o2", "never_intubated_room_air"):
        if random.random() < P_ACC_EXTUB:
            # eligible IMV hours
            imv_times = []
            for t in times:
                if truth.extub_time is None:
                    imv_times.append(t)
                else:
                    if t < truth.extub_time or (truth.reintub_time is not None and t >= truth.reintub_time):
                        imv_times.append(t)
            if len(imv_times) > 12:
                window = imv_times[6:-6] if len(imv_times) > 20 else imv_times
                accidental_time = random.choice(window)

                severe = truth.scenario in ("fail_niv_7d_then_imv", "prolonged_imv_or_trach")
                base_p_good = 0.3 if severe or shock_any else 0.6
                accidental_path_deteriorate = random.random() > base_p_good

    # Baselines
    hr_base = clamp(np.random.normal(92, 15), 50, 140)
    rr_base = clamp(np.random.normal(20, 5), 8, 40)
    map_base = clamp(np.random.normal(75, 12), 40, 120)
    spo2_base = clamp(np.random.normal(95, 2), 80, 100)
    temp_base = clamp(np.random.normal(37.2, 0.5), 35.0, 40.0)
    glu_base = clamp(np.random.normal(150, 35), 60, 350)

    rows = []
    prev_device = None

    for i, t in enumerate(times):
        # Base support type from scenario truth
        if truth.scenario.startswith("never_intubated"):
            support_type = "NIV" if (truth.scenario == "never_intubated_o2" and demand[i] > 1.05 and random.random() < 0.15) else "O2"
        else:
            if truth.extub_time is None:
                support_type = "IMV"
            else:
                if t < truth.extub_time:
                    support_type = "IMV"
                else:
                    if truth.reintub_time is not None and t >= truth.reintub_time:
                        support_type = "IMV"
                    else:
                        support_type = "NIV" if truth.scenario in ("extub_niv_short", "fail_niv_7d_then_imv") else "O2"

        resp_device = None
        mode = None
        fio2_equiv = None
        o2_flow = None

        # Accidental extub window (1 hour) + possible reintubation in next 1–3 hours
        in_acc_window = accidental_time is not None and (accidental_time <= t < accidental_time + pd.Timedelta(hours=1))
        reintubation_after_acc = accidental_time is not None and accidental_path_deteriorate and (
            accidental_time + pd.Timedelta(hours=1) <= t < accidental_time + pd.Timedelta(hours=4)
        )

        if in_acc_window:
            # suddenly off ventilator: oxygen/NIV/room air depending on tolerance
            if shock_any or truth.scenario in ("fail_niv_7d_then_imv", "prolonged_imv_or_trach"):
                if random.random() < 0.3:
                    resp_device = "NIV"
                    mode = "NIV"
                    fio2_equiv = int(clamp(np.random.normal(60, 8), 40, 100))
                else:
                    resp_device = "FACE_MASK"
                    o2_flow = 8
                    mode = "FACE_MASK"
                    fio2_equiv = o2_flow_to_fio2_equiv_pct(o2_flow)
            else:
                rtol = random.random()
                if rtol < 0.10:
                    resp_device = "ROOM_AIR"
                    mode = "ROOM_AIR"
                    fio2_equiv = 21
                elif rtol < 0.60:
                    resp_device = "NASAL_CANNULA"
                    o2_flow = 3
                    mode = "NASAL_CANNULA"
                    fio2_equiv = o2_flow_to_fio2_equiv_pct(o2_flow)
                else:
                    resp_device = "FACE_MASK"
                    o2_flow = 6
                    mode = "FACE_MASK"
                    fio2_equiv = o2_flow_to_fio2_equiv_pct(o2_flow)

        elif reintubation_after_acc:
            # deterioration -> reintubate on AC
            resp_device = "IMV"
            mode = "AC"
            fio2_equiv = int(clamp(np.random.normal(70, 8), 40, 100))

        else:
            if support_type == "IMV":
                resp_device = "IMV"
                if truth.extub_time is not None and (truth.reintub_time is None or t < truth.reintub_time):
                    hours_to_extub = (truth.extub_time - t).total_seconds() / 3600
                    wean_stage = clamp(1.0 - hours_to_extub / 96, 0.0, 1.0)
                else:
                    wean_stage = 0.0

                if t in sbt_hours:
                    mode = "DIRECT_OXYGEN"
                    fio2_equiv = int(clamp(np.random.normal(40, 6), 30, 60))
                else:
                    # IMV mode hierarchy by weaning stage
                    r = random.random()
                    if wean_stage < 0.3:
                        mode = "AC" if r < 0.55 else ("SIMV" if r < 0.90 else "PSV")
                    elif wean_stage < 0.7:
                        if r < 0.25:
                            mode = "AC"
                        elif r < 0.55:
                            mode = "SIMV"
                        elif r < 0.85:
                            mode = "PSV"
                        else:
                            mode = "CPAP"
                    else:
                        if r < 0.10:
                            mode = "AC"
                        elif r < 0.30:
                            mode = "SIMV"
                        elif r < 0.70:
                            mode = "PSV"
                        else:
                            mode = "CPAP"

                    if truth.extub_time is not None and (truth.reintub_time is None or t < truth.reintub_time):
                        trend = clamp(((truth.extub_time - t).total_seconds() / 3600) / 72, 0, 1)
                    else:
                        trend = 1.0
                    fio2_mean = 70 * trend + 40 * (1 - trend)
                    fio2_equiv = int(clamp(np.random.normal(fio2_mean, 8), 40, 100))

            elif support_type == "NIV":
                resp_device = "NIV"
                mode = "NIV"
                fio2_equiv = int(clamp(np.random.normal(55, 10), 40, 100))

            else:
                delta = float(demand[i] - (demand[i - 1] if i > 0 else demand[i]))
                resp_device, flow = _demand_to_device_flow_directional(float(demand[i]), prev_device, delta)
                mode = resp_device
                if resp_device == "ROOM_AIR":
                    fio2_equiv = 21
                else:
                    o2_flow = int(flow) if flow is not None else None
                    fio2_equiv = o2_flow_to_fio2_equiv_pct(int(o2_flow))

        prev_device = resp_device

        # Vitals
        hr = np.random.normal(hr_base + (8 if shock_any else 0) + (5 if resp_device == "IMV" else 0), 8)
        rr = np.random.normal(rr_base + (3 if resp_device in ("NIV", "FACE_MASK", "FMWR", "NASAL_CANNULA") else 0), 3)
        spo2 = np.random.normal(spo2_base - (2 if resp_device == "IMV" else 0), 1.5)
        map_ = np.random.normal(map_base - (8 if shock_any else 0), 10)
        pp = clamp(np.random.normal(45, 12), 25, 80)
        sbp = map_ + (2 / 3) * pp
        dbp = map_ - (1 / 3) * pp
        temp = np.random.normal(temp_base + (0.3 if resp_device == "IMV" else 0), 0.25)
        glu = np.random.normal(glu_base + (15 if random.random() < 0.10 else 0), 20)

        rows.append({
            "icu_stay_id": icu_stay_id,
            "chart_time": t,
            "resp_device": resp_device,
            "mode": mode,
            "fio2_equiv_pct": int(clamp(fio2_equiv, 21, 100)),
            "o2_flow_lpm": o2_flow,
            "hr_bpm": int(clamp(hr, 30, 220)),
            "rr_bpm": int(clamp(rr, 4, 80)),
            "spo2_pct": int(clamp(spo2, 50, 100)),
            "map_mmhg": int(clamp(map_, 25, 160)),
            "sbp_mmhg": int(clamp(sbp, 40, 260)),
            "dbp_mmhg": int(clamp(dbp, 20, 180)),
            "temp_c": round(clamp(temp, 34.0, 41.5), 1),
            "glucose_mgdl": int(clamp(glu, 30, 600)),
        })

    df = pd.DataFrame(rows)

    # SBT failures: some DIRECT_OXYGEN episodes revert to CPAP/AC later in the block
    for ep in sbt_episodes:
        if random.random() < 0.30:
            ep_sorted = sorted(ep)
            mask = df["chart_time"].isin(ep_sorted)
            idxs = df.index[mask].tolist()
            if len(idxs) >= 2:
                cut = idxs[int(len(idxs) / 2):]
                for j in cut:
                    df.at[j, "mode"] = weighted_choice([("CPAP", 0.6), ("AC", 0.4)])
                    df.at[j, "fio2_equiv_pct"] = int(clamp(df.at[j, "fio2_equiv_pct"] + random.randint(5, 15), 40, 100))

    return df

# =============================================================================
# MV hourly
# =============================================================================
def generate_mv_hourly_from_vitals(icu_stay_id, truth: StayTruth, vitals_clean: pd.DataFrame, mv_data_available: int):
    rows = []
    for _, r in vitals_clean.iterrows():
        t = r["chart_time"]
        resp_device = r["resp_device"]
        fio2 = r["fio2_equiv_pct"]

        peep = pip = tv = rr_set = None

        if mv_data_available == 0:
            # YEKATIT: keep resp_device/fio2_pct but no vent-setting charting
            pass
        else:
            if resp_device == "NIV":
                peep = round(clamp(np.random.normal(6, 2), 4, 12), 1)
                pip = round(clamp(np.random.normal(18, 4), 12, 30), 1)
                rr_set = int(clamp(np.random.normal(14, 4), 8, 24))
            elif resp_device == "IMV":
                if r.get("mode") == "DIRECT_OXYGEN" and random.random() < 0.50:
                    pass
                else:
                    if truth.extub_time is not None and (truth.reintub_time is None or t < truth.reintub_time):
                        trend = clamp(((truth.extub_time - t).total_seconds() / 3600) / 72, 0, 1)
                    else:
                        trend = 1.0
                    peep_mean = 10 * trend + 6 * (1 - trend)
                    pip_mean = 26 * trend + 18 * (1 - trend)
                    peep = round(clamp(np.random.normal(peep_mean, 1.5), 4, 16), 1)
                    pip = round(clamp(np.random.normal(pip_mean, 3), 12, 40), 1)
                    tv = int(clamp(np.random.normal(450, 60), 250, 650))
                    rr_set = int(clamp(np.random.normal(16, 4), 8, 28))

        rows.append({
            "icu_stay_id": icu_stay_id,
            "chart_time": t,
            "resp_device": resp_device,  # must not be missing
            "fio2_pct": fio2,            # must not be missing
            "peep": peep,
            "pip_cmH2O": pip,
            "tv_set_ml": tv,
            "rr_set": rr_set,
        })
    return pd.DataFrame(rows)

# =============================================================================
# Labs / micro / fluids
# =============================================================================
def generate_routine_lab_events(icu_stay_id, truth: StayTruth, shock_any=0, vap_any=0):
    rows = []
    base_times = list(q3d_range(truth.admit + pd.Timedelta(hours=6), truth.discharge))
    extra_times = []
    if shock_any == 1 and len(base_times) > 0 and random.random() < 0.6:
        for t in base_times:
            if random.random() < 0.25:
                extra_times.append(t + pd.Timedelta(hours=random.choice([12, 18, 24])))
    lab_times = sorted(set(base_times + extra_times))
    if not lab_times:
        return pd.DataFrame(columns=["icu_stay_id", "lab_time", "panel", "test_name", "value", "unit"])

    cr_base = clamp(np.random.lognormal(mean=np.log(1.0), sigma=0.35), 0.4, 6.0)
    wbc_base = clamp(np.random.normal(9, 2.5), 3, 18)
    inr_base = clamp(np.random.normal(1.2, 0.2), 0.9, 2.5)

    for t in lab_times:
        if random.random() < 0.08:
            continue

        wbc = round(clamp(np.random.normal(wbc_base, 3.0), 1.5, 40), 1)
        hgb = round(clamp(np.random.normal(10.5, 1.8), 5.5, 18), 1)
        plt = int(clamp(np.random.normal(220, 90), 10, 900))
        rows += [
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "CBC", "test_name": "WBC", "value": wbc, "unit": "10^9/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "CBC", "test_name": "HGB", "value": hgb, "unit": "g/dL"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "CBC", "test_name": "PLT", "value": plt, "unit": "10^9/L"},
        ]

        esr = int(clamp(np.random.normal(35 if vap_any else 25, 15), 1, 120))
        rows.append({"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "ESR", "test_name": "ESR", "value": esr, "unit": "mm/hr"})

        na = int(clamp(np.random.normal(138, 4), 120, 160))
        k = round(clamp(np.random.normal(4.1, 0.6), 2.0, 7.0), 1)
        cl_ = int(clamp(np.random.normal(102, 5), 85, 125))
        hco3 = int(clamp(np.random.normal(24, 4), 10, 40))
        rows += [
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "ELECTROLYTES", "test_name": "Na", "value": na, "unit": "mmol/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "ELECTROLYTES", "test_name": "K", "value": k, "unit": "mmol/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "ELECTROLYTES", "test_name": "Cl", "value": cl_, "unit": "mmol/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "ELECTROLYTES", "test_name": "HCO3", "value": hco3, "unit": "mmol/L"},
        ]

        bun = int(clamp(np.random.normal(22 + 10 * shock_any, 12), 2, 180))
        cr = round(clamp(np.random.normal(cr_base + 0.2 * shock_any, 0.35), 0.2, 12), 2)
        rows += [
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "RFT", "test_name": "BUN", "value": bun, "unit": "mg/dL"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "RFT", "test_name": "Creatinine", "value": cr, "unit": "mg/dL"},
        ]

        ast = int(clamp(np.random.normal(40 + 25 * shock_any, 30), 5, 1200))
        alt = int(clamp(np.random.normal(45 + 20 * shock_any, 35), 5, 1200))
        tbil = round(clamp(np.random.normal(1.0 + 0.6 * shock_any, 0.7), 0.1, 25), 1)
        rows += [
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "LFT", "test_name": "AST", "value": ast, "unit": "U/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "LFT", "test_name": "ALT", "value": alt, "unit": "U/L"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "LFT", "test_name": "Total_Bilirubin", "value": tbil, "unit": "mg/dL"},
        ]

        inr = round(clamp(np.random.normal(inr_base + 0.15 * shock_any, 0.25), 0.8, 6.0), 2)
        pt = round(clamp(np.random.normal(12 + (inr - 1.0) * 6, 2.0), 8, 60), 1)
        aptt = round(clamp(np.random.normal(32 + 6 * shock_any, 10), 18, 180), 1)
        rows += [
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "COAG", "test_name": "INR", "value": inr, "unit": ""},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "COAG", "test_name": "PT", "value": pt, "unit": "sec"},
            {"icu_stay_id": icu_stay_id, "lab_time": t, "panel": "COAG", "test_name": "aPTT", "value": aptt, "unit": "sec"},
        ]

    return pd.DataFrame(rows)

def generate_micro_orders(icu_stay_id, truth: StayTruth, shock_any=0, vap_any=0):
    rows = []
    los_days = (truth.discharge - truth.admit).total_seconds() / 86400.0
    p_blood = 0.08 + 0.18 * shock_any
    p_sputum = 0.06 + 0.22 * vap_any
    p_urine = 0.06 + (0.05 if los_days > 14 else 0.0)

    if random.random() < p_blood:
        ot = rand_time_between(truth.admit + pd.Timedelta(hours=6), truth.discharge - pd.Timedelta(hours=24))
        rt = ot + pd.Timedelta(hours=random.randint(24, 72))
        result = weighted_choice([("NEGATIVE", 0.70), ("POSITIVE", 0.20), ("CONTAMINANT", 0.10)])
        organism = None
        if result == "POSITIVE":
            organism = weighted_choice([
                ("E. coli", 0.20),
                ("Klebsiella pneumoniae", 0.20),
                ("Staphylococcus aureus", 0.25),
                ("Pseudomonas aeruginosa", 0.15),
                ("Enterococcus", 0.20),
            ])
        rows.append({"icu_stay_id": icu_stay_id, "order_time": ot, "specimen": "BLOOD",
                     "test": "CULTURE", "result_time": rt, "result": result, "organism": organism, "notes": None})

    if random.random() < p_sputum:
        ot = rand_time_between(truth.admit + pd.Timedelta(hours=12), truth.discharge - pd.Timedelta(hours=24))
        rt = ot + pd.Timedelta(hours=random.randint(24, 72))
        result = weighted_choice([("NEGATIVE", 0.55), ("POSITIVE", 0.35), ("CONTAMINANT", 0.10)])
        organism = None
        if result == "POSITIVE":
            organism = weighted_choice([
                ("Pseudomonas aeruginosa", 0.25),
                ("Klebsiella pneumoniae", 0.20),
                ("Acinetobacter", 0.15),
                ("MRSA", 0.15),
                ("Streptococcus pneumoniae", 0.25),
            ])
        rows.append({"icu_stay_id": icu_stay_id, "order_time": ot, "specimen": "SPUTUM",
                     "test": "CULTURE", "result_time": rt, "result": result, "organism": organism, "notes": None})

    if random.random() < p_urine:
        ot = rand_time_between(truth.admit + pd.Timedelta(hours=12), truth.discharge - pd.Timedelta(hours=24))
        rt = ot + pd.Timedelta(hours=random.randint(24, 72))
        result = weighted_choice([("NEGATIVE", 0.65), ("POSITIVE", 0.28), ("CONTAMINANT", 0.07)])
        organism = None
        if result == "POSITIVE":
            organism = weighted_choice([("E. coli", 0.55), ("Klebsiella", 0.15), ("Enterococcus", 0.15), ("Proteus", 0.15)])
        rows.append({"icu_stay_id": icu_stay_id, "order_time": ot, "specimen": "URINE",
                     "test": "CULTURE", "result_time": rt, "result": result, "organism": organism, "notes": None})

    if random.random() < (0.03 + 0.05 * vap_any):
        ot = rand_time_between(truth.admit + pd.Timedelta(hours=24), truth.discharge - pd.Timedelta(hours=12))
        rt = ot + pd.Timedelta(hours=random.randint(4, 12))
        result = weighted_choice([("NEGATIVE", 0.80), ("POSITIVE", 0.18), ("INVALID", 0.02)])
        rows.append({"icu_stay_id": icu_stay_id, "order_time": ot, "specimen": "SPUTUM",
                     "test": "GENEXPERT", "result_time": rt, "result": result,
                     "organism": "MTB" if result == "POSITIVE" else None, "notes": None})

    return pd.DataFrame(rows)

def generate_fluid_balance_daily(icu_stay_id, truth: StayTruth, aki_any=0, shock_any=0):
    rows = []
    t = truth.admit
    day_idx = 0
    while t < truth.discharge:
        day_start = t
        day_end = min(truth.discharge, day_start + pd.Timedelta(days=1))
        if random.random() < 0.12:
            t = day_end
            day_idx += 1
            continue
        intake = int(clamp(np.random.normal(2600 + 700 * shock_any, 600), 800, 9000))
        output = int(clamp(np.random.normal(2400 - 900 * aki_any, 700), 200, 9000))
        net = intake - output
        if random.random() < 0.05:
            net += random.choice([-500, -300, -200, 200, 300, 500, 800, -800])
        rows.append({
            "icu_stay_id": icu_stay_id,
            "day_index": day_idx,
            "day_start_time": day_start,
            "day_end_time": day_end,
            "intake_ml": intake,
            "output_ml": output,
            "net_balance_ml": net,
        })
        t = day_end
        day_idx += 1
    if not rows:
        return pd.DataFrame(columns=[
            "icu_stay_id", "day_index", "day_start_time", "day_end_time",
            "intake_ml", "output_ml", "net_balance_ml"
        ])
    return pd.DataFrame(rows)

# =============================================================================
# Medications
# =============================================================================
FORMULARY = [
    {"name": "Pantoprazole", "route": "IV", "freq": "daily", "base_p": 0.65},
    {"name": "Heparin", "route": "SC", "freq": "q12h", "base_p": 0.55},
    {"name": "Enoxaparin", "route": "SC", "freq": "daily", "base_p": 0.25},
    {"name": "Ceftriaxone", "route": "IV", "freq": "daily", "base_p": 0.25},
    {"name": "Piperacillin-Tazobactam", "route": "IV", "freq": "q6h", "base_p": 0.18},
    {"name": "Vancomycin", "route": "IV", "freq": "q12h", "base_p": 0.10},
    {"name": "Propofol", "route": "IV", "freq": "infusion", "base_p": 0.18},
    {"name": "Midazolam", "route": "IV", "freq": "infusion", "base_p": 0.08},
    {"name": "Fentanyl", "route": "IV", "freq": "infusion", "base_p": 0.15},
    {"name": "Norepinephrine", "route": "IV", "freq": "infusion", "base_p": 0.10},
    {"name": "Hydrocortisone", "route": "IV", "freq": "q6h", "base_p": 0.08},
]

def _choose_initial_meds(stay_vars: dict, vitals_clean: pd.DataFrame) -> list[dict]:
    shock_any = int(stay_vars.get("shock_any", 0))
    steroid_any = int(stay_vars.get("steroid_any", 0))
    vap_any = int(stay_vars.get("vap_any", 0))
    any_imv = (vitals_clean["resp_device"] == "IMV").any()

    chosen = []
    for med in FORMULARY:
        p = med["base_p"]
        if med["name"] == "Norepinephrine":
            p = 0.60 if shock_any else 0.05
        if med["name"] == "Hydrocortisone":
            p = 0.35 if steroid_any else 0.05
        if med["name"] in ("Propofol", "Midazolam", "Fentanyl"):
            p = p * (1.8 if any_imv else 0.4)
        if med["name"] in ("Ceftriaxone", "Piperacillin-Tazobactam", "Vancomycin"):
            p = p * (1.8 if vap_any else 0.9)
        if random.random() < clamp(p, 0.0, 0.95):
            chosen.append(med)

    names = {m["name"] for m in chosen}
    if "Heparin" in names and "Enoxaparin" in names and random.random() < 0.8:
        chosen = [m for m in chosen if m["name"] != "Enoxaparin"]
    return chosen

def generate_med_orders_and_admin(icu_stay_id: str, truth: StayTruth, stay_vars: dict, vitals_clean: pd.DataFrame):
    orders = []
    admin = []
    admit = truth.admit
    discharge = truth.discharge

    meds = _choose_initial_meds(stay_vars, vitals_clean)

    for med in meds:
        ot = admit + pd.Timedelta(hours=random.randint(0, 6))
        start = ot
        stop = None
        if med["freq"] == "infusion":
            if random.random() < 0.35:
                stop = start + pd.Timedelta(hours=random.randint(12, 72))
        else:
            if random.random() < 0.25:
                stop = start + pd.Timedelta(days=random.randint(2, 7))
        if stop is not None and stop >= discharge:
            stop = None

        dose = weighted_choice([("low", 0.4), ("standard", 0.45), ("high", 0.15)])
        orders.append({
            "icu_stay_id": icu_stay_id,
            "order_time": ot,
            "ordered_by": "PHYSICIAN",
            "med_name": med["name"],
            "dose": dose,
            "route": med["route"],
            "frequency": med["freq"],
            "start_time": start,
            "stop_time": stop,
            "order_status": "ACTIVE" if stop is None else "DISCONTINUED",
        })

        if med["freq"] == "infusion":
            step_h = 4
        elif med["freq"] == "q6h":
            step_h = 6
        elif med["freq"] == "q12h":
            step_h = 12
        else:
            step_h = 24

        end_time = stop if stop is not None else discharge
        t = start
        while t < end_time:
            given = 1
            reason = None
            if random.random() < 0.06:
                given = 0
                reason = weighted_choice([("patient unavailable", 0.4), ("low BP", 0.2), ("contraindicated", 0.2), ("other", 0.2)])
            admin.append({
                "icu_stay_id": icu_stay_id,
                "admin_time": t,
                "signed_by": "NURSING",
                "med_name": med["name"],
                "dose_given": dose,
                "given_flag": given,
                "reason_not_given": reason,
            })
            t = t + pd.Timedelta(hours=step_h)

    # additional orders around extub/reintub times
    for et in [x for x in [truth.extub_time, truth.reintub_time] if x is not None]:
        if random.random() < 0.35:
            med = weighted_choice([(m, m["base_p"]) for m in FORMULARY])
            ot = et + pd.Timedelta(hours=random.randint(-6, 6))
            if ot < admit:
                ot = admit
            if ot >= discharge:
                continue
            dose = weighted_choice([("low", 0.4), ("standard", 0.45), ("high", 0.15)])
            orders.append({
                "icu_stay_id": icu_stay_id,
                "order_time": ot,
                "ordered_by": "PHYSICIAN",
                "med_name": med["name"],
                "dose": dose,
                "route": med["route"],
                "frequency": med["freq"],
                "start_time": ot,
                "stop_time": None,
                "order_status": "ACTIVE",
            })

    return pd.DataFrame(orders), pd.DataFrame(admin)

def _active_meds_at_time(med_orders: pd.DataFrame, t: pd.Timestamp) -> list[str]:
    if len(med_orders) == 0:
        return []
    mo = med_orders.copy()
    mo["stop_time"] = pd.to_datetime(mo["stop_time"], errors="coerce")
    active = mo[(mo["start_time"] <= t) & ((mo["stop_time"].isna()) | (mo["stop_time"] > t))]
    return list(dict.fromkeys(active["med_name"].tolist()))

def _new_or_changed_meds_last_24h(med_orders: pd.DataFrame, t: pd.Timestamp) -> list[str]:
    if len(med_orders) == 0:
        return []
    t0 = t - pd.Timedelta(hours=24)
    recent = med_orders[(med_orders["order_time"] >= t0) & (med_orders["order_time"] <= t)]
    return list(dict.fromkeys(recent["med_name"].tolist()))

def _mar_events_in_window(med_admin: pd.DataFrame, t0: pd.Timestamp, t1: pd.Timestamp) -> tuple[list[str], list[str]]:
    if len(med_admin) == 0:
        return [], []
    ma = med_admin[(med_admin["admin_time"] >= t0) & (med_admin["admin_time"] < t1)]
    given = ma[ma["given_flag"] == 1]["med_name"].tolist()
    missed = ma[ma["given_flag"] == 0]["med_name"].tolist()
    return list(dict.fromkeys(given)), list(dict.fromkeys(missed))

def _select_key_meds_for_nursing(given: list[str], missed: list[str]) -> tuple[list[str], list[str]]:
    priority = ["Norepinephrine", "Propofol", "Midazolam", "Fentanyl",
                "Ceftriaxone", "Piperacillin-Tazobactam", "Vancomycin"]
    given_key = [m for m in given if m in priority][:2] or given[:2]
    missed_key = [m for m in missed if m in priority][:1] or missed[:1]
    return given_key, missed_key

# =============================================================================
# Notes
# =============================================================================
def render_physician_note(truth: StayTruth, vitals_obs: pd.DataFrame, mv_obs: pd.DataFrame,
                          med_orders: pd.DataFrame, note_time: pd.Timestamp):
    vs = vitals_obs[vitals_obs["chart_time"] <= note_time].tail(1)
    if len(vs) == 0:
        return "ICU Progress Note\n\nInterval events: chart unavailable.\n"
    v = vs.iloc[0].to_dict()
    mv = mv_obs[mv_obs["chart_time"] <= note_time].tail(1)
    m = mv.iloc[0].to_dict() if len(mv) else {}

    if random.random() < P_QUALITATIVE_ONLY:
        resp_line = f"Resp: {v.get('resp_device','UNKNOWN')} on minimal support."
    else:
        if v.get("resp_device") in ("IMV", "NIV"):
            resp_line = f"Resp: {v.get('resp_device')} mode {v.get('mode')} FiO2 {v.get('fio2_equiv_pct')}%"
            if v.get("resp_device") == "IMV":
                resp_line += f", PEEP {m.get('peep')}, PIP {m.get('pip_cmH2O')}, TV {m.get('tv_set_ml')}"
            else:
                resp_line += f", PEEP {m.get('peep')}, PIP {m.get('pip_cmH2O')}"
            resp_line += "."
        else:
            if v.get("resp_device") == "ROOM_AIR":
                resp_line = "ROOM_AIR."
            else:
                resp_line = f"Resp: {v.get('resp_device')} {v.get('o2_flow_lpm')} L/min (FiO2~{v.get('fio2_equiv_pct')}%)."

    interval = []
    if truth.extub_time is not None and truth.extub_time <= note_time and random.random() > P_OMIT_EVENT:
        when = "overnight" if random.random() < P_TIME_AMBIGUOUS else str(truth.extub_time)
        interval.append(f"Extubated {when}.")
    if truth.reintub_time is not None and truth.reintub_time <= note_time and random.random() > P_OMIT_EVENT:
        when = "this morning" if random.random() < P_TIME_AMBIGUOUS else str(truth.reintub_time)
        interval.append(f"Reintubated {when} for respiratory failure.")
    if truth.death_time is not None and truth.death_time <= note_time and random.random() > 0.5:
        interval.append("Patient expired; see event note.")
    if random.random() < P_COPY_FORWARD_NOTE:
        interval.append("No significant interval changes (per prior note).")

    # Accidental extubation mention (heuristic from last 24h device pattern)
    last_24 = vitals_obs[(vitals_obs["chart_time"] <= note_time) &
                         (vitals_obs["chart_time"] > note_time - pd.Timedelta(hours=24))]
    if len(last_24) >= 3:
        dev = last_24["resp_device"].astype(str).tolist()
        for i in range(len(dev) - 2):
            if dev[i] == "IMV" and dev[i + 1] not in ("IMV", "NIV") and dev[i + 2] == "IMV":
                interval.append("Unplanned extubation with prompt reintubation on AC mode.")
                break
        else:
            for i in range(len(dev) - 1):
                if dev[i] == "IMV" and dev[i + 1] in ("NASAL_CANNULA", "FACE_MASK", "ROOM_AIR"):
                    interval.append("Unplanned extubation, remained stable on oxygen without immediate reintubation.")
                    break

    interval_text = " ".join(interval) if interval else "No major overnight events documented."

    secretions = weighted_choice([("scant", 0.25), ("moderate", 0.45), ("copious", 0.20), ("thick/purulent", 0.10)])
    wob = weighted_choice([("no increased work of breathing", 0.55), ("mild increased WOB", 0.25), ("marked increased WOB", 0.20)])
    cough = weighted_choice([("strong", 0.30), ("moderate", 0.45), ("weak", 0.25)])
    agitation = weighted_choice([("calm", 0.55), ("restless", 0.25), ("agitated", 0.20)])
    gcs = int(clamp(np.random.normal(11, 3), 3, 15))

    active_meds = _active_meds_at_time(med_orders, note_time)
    recent_meds = _new_or_changed_meds_last_24h(med_orders, note_time)
    meds_line = "Meds: " + (", ".join(active_meds) if active_meds else "none documented.")
    if recent_meds:
        meds_line += " New/changed in last 24h: " + ", ".join(recent_meds) + "."

    return (
        "ICU Progress Note\n"
        f"Date/Time: {note_time}\n\n"
        f"Interval events: {interval_text}\n\n"
        "Objective:\n"
        f"- {resp_line}\n"
        f"- HR {v.get('hr_bpm')} RR {v.get('rr_bpm')} SpO2 {v.get('spo2_pct')} MAP {v.get('map_mmhg')}\n"
        f"- GCS: {gcs} ({agitation})\n"
        f"- Secretions: {secretions}\n"
        f"- Work of breathing: {wob}\n"
        f"- Cough: {cough}\n"
        f"- {meds_line}\n\n"
        "Assessment/Plan:\n"
        "- Acute respiratory failure: continue weaning as tolerated; consider SBT.\n"
        "- Adjust medications per above orders; continue monitoring for response and adverse effects.\n"
    )

def render_nursing_note(truth: StayTruth, vitals_obs: pd.DataFrame, note_time: pd.Timestamp, shift: str,
                        fluid_row: dict | None, med_admin: pd.DataFrame):
    vs = vitals_obs[vitals_obs["chart_time"] <= note_time].tail(1)
    if len(vs) == 0:
        return f"Nursing Shift Note ({shift})\nChart unavailable.\n"
    v = vs.iloc[0].to_dict()

    suction = weighted_choice([("q4h", 0.35), ("q2h", 0.35), ("frequent", 0.20), ("minimal", 0.10)])
    secretions = weighted_choice([("thin", 0.35), ("thick", 0.35), ("blood-tinged", 0.10), ("purulent", 0.20)])
    agitation = weighted_choice([("calm", 0.50), ("intermittently agitated", 0.30), ("agitated", 0.20)])
    wob = weighted_choice([("no increased WOB", 0.55), ("mild increased WOB", 0.25), ("increased WOB", 0.20)])

    if v.get("resp_device") in ("IMV", "NIV"):
        resp = f"{v.get('resp_device')} {v.get('mode')} FiO2 {v.get('fio2_equiv_pct')}%."
    else:
        if v.get("resp_device") == "ROOM_AIR":
            resp = "ROOM_AIR."
        else:
            resp = f"{v.get('resp_device')} {v.get('o2_flow_lpm')} L/min (FiO2~{v.get('fio2_equiv_pct')}%)."

    events = []
    if truth.extub_time is not None and abs((truth.extub_time - note_time).total_seconds()) < 12 * 3600 and random.random() > P_OMIT_EVENT:
        events.append("Extubation discussed with team.")
    if truth.reintub_time is not None and abs((truth.reintub_time - note_time).total_seconds()) < 12 * 3600 and random.random() > P_OMIT_EVENT:
        events.append("Provider notified of respiratory distress.")
    event_text = " ".join(events) if events else "No acute events."

    io_text = "I/O: not charted."
    if fluid_row is not None:
        intake = fluid_row["intake_ml"]
        output = fluid_row["output_ml"]
        net = fluid_row["net_balance_ml"]
        if random.random() < P_IO_NOTE_MISMATCH and pd.notna(net):
            net = net + random.choice([-400, -200, 200, 400, 700, -700])
        if random.random() < 0.20:
            io_text = f"I/O: Net {net:+.0f} mL (24h)." if pd.notna(net) else "I/O: Net not available."
        else:
            if pd.notna(intake) and pd.notna(output) and pd.notna(net):
                io_text = f"I/O (24h): Intake {intake:.0f} mL, Output {output:.0f} mL, Net {net:+.0f} mL."
            else:
                io_text = "I/O (24h): not fully charted."

    t0 = note_time - pd.Timedelta(hours=12)
    given, missed = _mar_events_in_window(med_admin, t0, note_time)
    given_key, missed_key = _select_key_meds_for_nursing(given, missed)
    if given_key or missed_key:
        parts = []
        if given_key:
            parts.append("Given: " + ", ".join(given_key))
        if missed_key:
            parts.append("Held/Not given: " + ", ".join(missed_key))
        mar_line = "MAR (key meds): " + "; ".join(parts)
    else:
        mar_line = "MAR: no key medication events this shift."

    return (
        f"Nursing Shift Note ({shift})\n"
        f"Date/Time: {note_time}\n"
        f"Nursing diagnoses: impaired gas exchange; risk for infection.\n"
        f"Resp: {resp}\n"
        f"Vitals: HR {v.get('hr_bpm')} RR {v.get('rr_bpm')} SpO2 {v.get('spo2_pct')} MAP {v.get('map_mmhg')}\n"
        f"WOB: {wob}\n"
        f"Suctioning: {suction}; secretions: {secretions}\n"
        f"Neuro/Behavior: {agitation}\n"
        f"{io_text}\n"
        f"{mar_line}\n"
        f"Events: {event_text}\n"
    )

def generate_notes(icu_stay_id, truth: StayTruth, vitals_obs: pd.DataFrame, mv_obs: pd.DataFrame,
                   fluids_df: pd.DataFrame, med_orders: pd.DataFrame, med_admin: pd.DataFrame):
    rows = []

    # physician daily at 09:00
    day = truth.admit.floor("D")
    while day < truth.discharge:
        t = day + pd.Timedelta(hours=9)
        if truth.admit <= t < truth.discharge:
            rows.append({
                "note_id": uid("note"),
                "icu_stay_id": icu_stay_id,
                "note_time": t,
                "author_type": "PHYSICIAN",
                "note_type": "ICU Progress Note",
                "shift": None,
                "text": render_physician_note(truth, vitals_obs, mv_obs, med_orders, t),
            })
        day += pd.Timedelta(days=1)

    # nursing shifts at 07:00/19:00
    day = truth.admit.floor("D")
    while day < truth.discharge:
        for hr, shift in [(7, "DAY"), (19, "NIGHT")]:
            t = day + pd.Timedelta(hours=hr)
            if truth.admit <= t < truth.discharge:
                fr = fluids_df[(fluids_df["icu_stay_id"] == icu_stay_id) &
                               (fluids_df["day_start_time"] <= t) &
                               (fluids_df["day_end_time"] > t)]
                fluid_row = fr.iloc[0].to_dict() if len(fr) else None
                rows.append({
                    "note_id": uid("note"),
                    "icu_stay_id": icu_stay_id,
                    "note_time": t,
                    "author_type": "NURSING",
                    "note_type": "Nursing Shift Note",
                    "shift": shift,
                    "text": render_nursing_note(truth, vitals_obs, t, shift, fluid_row, med_admin),
                })
        day += pd.Timedelta(days=1)

    return pd.DataFrame(rows)

# =============================================================================
# Noise functions
# =============================================================================
def apply_vitals_noise(vitals_df: pd.DataFrame, stays_df: pd.DataFrame | None = None) -> pd.DataFrame:
    df = _ensure_sorted(vitals_df.copy(), ["icu_stay_id", "chart_time"])

    shock_map = None
    if stays_df is not None and {"icu_stay_id", "shock_any"}.issubset(stays_df.columns):
        shock_map = stays_df.set_index("icu_stay_id")["shock_any"].to_dict()

    for icu_stay_id, g_idx in df.groupby("icu_stay_id", sort=False).groups.items():
        idx = np.array(list(g_idx), dtype=int)
        g = df.loc[idx].copy()
        times = g["chart_time"]
        shock_any = int(shock_map.get(icu_stay_id, 0)) if shock_map is not None else None

        # oxygen flow not charted on IMV/NIV
        g.loc[g["resp_device"].isin(["IMV", "NIV"]), "o2_flow_lpm"] = np.nan

        # additive noise
        for col, sd in [("temp_c", 0.05), ("map_mmhg", 1.5), ("sbp_mmhg", 2.0), ("dbp_mmhg", 2.0), ("glucose_mgdl", 3.0)]:
            g[col] = _additive_measurement_noise(g[col], sd=sd)

        # re-clip
        g["temp_c"] = _clip_float_series(g["temp_c"], 34.0, 41.5, decimals=1)
        for col, lo, hi in [("map_mmhg", 25, 160), ("sbp_mmhg", 40, 260), ("dbp_mmhg", 20, 180), ("glucose_mgdl", 30, 600)]:
            g[col] = _clip_int_series(g[col], lo, hi)

        # block gaps
        for col in _VITALS_NUM_COLS:
            n_blocks = _poisson_capped(lam=0.6, cap=3)
            for _ in range(n_blocks):
                start = _pick_start_index_weighted(times)
                L = _pick_block_len_hours()
                end = min(start + L, len(g))
                g.loc[g.index[start:end], col] = np.nan

        # shock-dependent slightly more missing temp/glucose
        if shock_any == 1:
            for col in ["temp_c", "glucose_mgdl"]:
                m = g[col].notna() & (np.random.rand(len(g)) < 0.03)
                g.loc[m, col] = np.nan

        # MNAR for extremes
        def mnar(col: str, extreme_mask: pd.Series, base: float, k: float):
            s = g[col]
            mask = s.notna()
            extreme = extreme_mask & mask
            if extreme.any():
                score = extreme.astype(float)
                p = _logistic(base + k * score)
                toss = np.random.rand(len(g)) < p
                g.loc[extreme & toss, col] = np.nan

        mnar("spo2_pct", g["spo2_pct"].astype(float) < 85, base=-3.0, k=2.0)
        mnar("map_mmhg", g["map_mmhg"].astype(float) < 55, base=-3.2, k=2.2)
        mnar("hr_bpm", g["hr_bpm"].astype(float) > 140, base=-3.2, k=2.0)
        mnar("rr_bpm", g["rr_bpm"].astype(float) > 35, base=-3.2, k=2.0)
        mnar("temp_c", g["temp_c"].astype(float) > 39.5, base=-3.0, k=2.2)

        # stuck sensor
        for col in ["hr_bpm", "spo2_pct", "map_mmhg", "rr_bpm"]:
            if random.random() < 0.02 and len(g) >= 4:
                L = random.randint(2, 8)
                start = random.randint(0, max(0, len(g) - L))
                end = start + L
                val = g.iloc[start][col]
                if pd.isna(val):
                    sub = g.iloc[start:end][col].dropna()
                    if len(sub) > 0:
                        val = sub.iloc[0]
                if not pd.isna(val):
                    g.loc[g.index[start:end], col] = val

        # spikes/outliers
        if random.random() < 0.01 and len(g) > 0:
            n_spikes = random.randint(1, 3)
            spike_rows = np.random.choice(np.arange(len(g)), size=n_spikes, replace=False)
            for sr in spike_rows:
                col = random.choice(["hr_bpm", "rr_bpm", "spo2_pct", "map_mmhg", "temp_c", "glucose_mgdl"])
                if col == "spo2_pct":
                    g.iloc[sr, g.columns.get_loc(col)] = random.choice([10, 40, 105])
                elif col == "temp_c":
                    g.iloc[sr, g.columns.get_loc(col)] = random.choice([30.0, 42.5])
                elif col == "glucose_mgdl":
                    g.iloc[sr, g.columns.get_loc(col)] = random.choice([10, 900])
                elif col == "map_mmhg":
                    g.iloc[sr, g.columns.get_loc(col)] = random.choice([0, 250, 500])
                else:
                    g.iloc[sr, g.columns.get_loc(col)] = random.choice([0, 300, 500])

        # enforce never-missing
        for col in _VITALS_NEVER_MISSING:
            if g[col].isna().any():
                g[col] = g[col].fillna(df.loc[idx, col].values)

        df.loc[idx, g.columns] = g.values

    return df

def apply_mv_noise(mv_df: pd.DataFrame, vitals_df: pd.DataFrame) -> pd.DataFrame:
    df = _ensure_sorted(mv_df.copy(), ["icu_stay_id", "chart_time"])
    vit = _ensure_sorted(vitals_df[["icu_stay_id", "chart_time", "resp_device"]].copy(), ["icu_stay_id", "chart_time"])
    df = df.merge(vit, on=["icu_stay_id", "chart_time"], how="left", suffixes=("", "_v"))
    df["resp_device"] = df["resp_device_v"].fillna(df["resp_device"])
    df = df.drop(columns=["resp_device_v"])

    for icu_stay_id, g_idx in df.groupby("icu_stay_id", sort=False).groups.items():
        idx = np.array(list(g_idx), dtype=int)
        g = df.loc[idx].copy()

        # delayed chart update after device change
        dev = g["resp_device"].astype(str)
        change = dev.ne(dev.shift(1)).fillna(False)
        change_positions = np.where(change.to_numpy())[0]

        for pos in change_positions:
            if pos == 0:
                continue
            lag_h = random.randint(2, 8)

            prev = g.iloc[:pos][_MV_VENT_COLS]
            last_valid_i = None
            for j in range(pos - 1, -1, -1):
                if prev.iloc[j].notna().any():
                    last_valid_i = j
                    break
            if last_valid_i is None:
                continue

            fill_vals = prev.iloc[last_valid_i].to_dict()
            end = min(pos + lag_h, len(g))
            block_idx = g.index[pos:end]
            for c in _MV_VENT_COLS:
                cur = g.loc[block_idx, c]
                g.loc[block_idx, c] = cur.where(cur.notna(), fill_vals.get(c, np.nan))

        # rare scale error
        if random.random() < 0.001:
            ventilated = g["resp_device"].isin(["IMV", "NIV"]) & g["pip_cmH2O"].notna()
            if ventilated.any():
                row_i = int(np.random.choice(np.where(ventilated.to_numpy())[0], size=1)[0])
                new_val = float(g.iloc[row_i]["pip_cmH2O"]) * 10.0
                g.iloc[row_i, g.columns.get_loc("pip_cmH2O")] = float(clamp(new_val, 0.0, 120.0))

        # enforce non-missing resp_device/fio2_pct
        if g["resp_device"].isna().any():
            g["resp_device"] = g["resp_device"].fillna(df.loc[idx, "resp_device"].values)
        if g["fio2_pct"].isna().any():
            g["fio2_pct"] = g["fio2_pct"].fillna(df.loc[idx, "fio2_pct"].values)

        df.loc[idx, g.columns] = g.values

    return df

def apply_lab_noise(lab_df: pd.DataFrame, stays_df: pd.DataFrame) -> pd.DataFrame:
    df = _ensure_sorted(lab_df.copy(), ["icu_stay_id", "lab_time"])
    m = stays_df.set_index("icu_stay_id")
    admit_map = m["admit_time"].to_dict()
    discharge_map = m["discharge_time"].to_dict()

    panel = df["panel"].astype(str)
    base_missing = np.random.uniform(0.05, 0.15)
    panel_factor = np.ones(len(df), dtype=float)
    panel_factor *= np.where(panel == "ELECTROLYTES", 0.7, 1.0)
    panel_factor *= np.where(panel.isin(["COAG", "LFT"]), 1.25, 1.0)
    p_miss = np.clip(base_missing * panel_factor, 0.01, 0.35)

    miss_mask = (df["value"].notna()) & (np.random.rand(len(df)) < p_miss)
    df.loc[miss_mask, "value"] = np.nan

    outlier_mask = np.random.rand(len(df)) < 0.005
    if outlier_mask.any():
        cr_mask = outlier_mask & (df["test_name"] == "Creatinine")
        df.loc[cr_mask, "value"] = np.where(np.random.rand(cr_mask.sum()) < 0.5, 0.1, 15.0)
        wbc_mask = outlier_mask & (df["test_name"] == "WBC")
        df.loc[wbc_mask, "value"] = np.where(np.random.rand(wbc_mask.sum()) < 0.5, 0.5, 80.0)
        inr_mask = outlier_mask & (df["test_name"] == "INR")
        df.loc[inr_mask, "value"] = np.where(np.random.rand(inr_mask.sum()) < 0.5, 0.7, 10.0)

    jitter_mask = np.random.rand(len(df)) < 0.10
    if jitter_mask.any():
        jitter_h = np.random.randint(-6, 7, size=int(jitter_mask.sum()))
        new_times = df.loc[jitter_mask, "lab_time"] + pd.to_timedelta(jitter_h, unit="h")
        icu_ids = df.loc[jitter_mask, "icu_stay_id"].tolist()
        clamped = []
        for t, sid in zip(new_times.tolist(), icu_ids):
            a = admit_map.get(sid, t)
            d = discharge_map.get(sid, t)
            if pd.notna(a) and t < a:
                t = a
            if pd.notna(d) and t >= d:
                t = d - pd.Timedelta(minutes=1)
            clamped.append(t)
        df.loc[jitter_mask, "lab_time"] = clamped

    return df

def apply_fluids_noise(fluids_df: pd.DataFrame) -> pd.DataFrame:
    df = _ensure_sorted(fluids_df.copy(), ["icu_stay_id", "day_start_time"])
    n = len(df)

    miss_days = np.random.rand(n) < 0.10
    if miss_days.any():
        choose_intake = np.random.rand(int(miss_days.sum())) < 0.5
        miss_idx = np.where(miss_days)[0]
        intake_idx = miss_idx[choose_intake]
        output_idx = miss_idx[~choose_intake]
        if len(intake_idx):
            df.loc[df.index[intake_idx], "intake_ml"] = np.nan
        if len(output_idx):
            df.loc[df.index[output_idx], "output_ml"] = np.nan

    net_mask = df["intake_ml"].isna() | df["output_ml"].isna()
    df.loc[net_mask, "net_balance_ml"] = np.nan

    mismatch_mask = (df["intake_ml"].notna() & df["output_ml"].notna() & (np.random.rand(n) < 0.08))
    if mismatch_mask.any():
        err = np.random.choice([-1, 1], size=int(mismatch_mask.sum())) * np.random.randint(200, 801, size=int(mismatch_mask.sum()))
        df.loc[mismatch_mask, "net_balance_ml"] = df.loc[mismatch_mask, "net_balance_ml"].astype(float) + err

    return df

def apply_micro_noise(micro_df: pd.DataFrame) -> pd.DataFrame:
    df = micro_df.copy()
    if len(df) == 0:
        return df
    df = _ensure_sorted(df, ["icu_stay_id", "order_time"])
    rt_mask = np.random.rand(len(df)) < 0.10
    df.loc[rt_mask, "result_time"] = pd.NaT

    if "notes" in df.columns:
        notes_null = df["notes"].isna()
        tweak_mask = notes_null & (np.random.rand(len(df)) < 0.07)
        if tweak_mask.any():
            new_vals = []
            for _ in range(int(tweak_mask.sum())):
                new_vals.append(weighted_choice([("INVALID", 0.45), ("CONTAMINANT", 0.35), ("NEGATIVE", 0.15), ("POSITIVE", 0.05)]))
            df.loc[tweak_mask, "result"] = new_vals
    return df

def apply_notes_noise(notes_df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    df = _ensure_sorted(notes_df.copy(), ["icu_stay_id", "note_time"])
    modified = np.zeros(len(df), dtype=bool)

    vague_phrases = ["overnight", "this morning", "earlier today", "yesterday", "recently"]
    for i in range(len(df)):
        if random.random() < 0.25:
            txt = df.at[i, "text"]
            if isinstance(txt, str) and _TIMESTAMP_RE.search(txt):
                df.at[i, "text"] = _TIMESTAMP_RE.sub(random.choice(vague_phrases), txt)
                modified[i] = True

    for (_, author_type), g in df.groupby(["icu_stay_id", "author_type"], sort=False):
        if len(g) < 2:
            continue
        if random.random() < 0.15:
            positions = g.index.to_list()
            src_pos = random.choice(positions[:-1])
            dst_pos = positions[positions.index(src_pos) + 1]
            df.at[dst_pos, "text"] = df.at[src_pos, "text"]
            modified[dst_pos] = True

    swaps = [
        ("Resp: IMV", "Resp: NIV"),
        ("Resp: NIV", "Resp: IMV"),
        ("ROOM_AIR", "NASAL_CANNULA"),
        ("NASAL_CANNULA", "FACE_MASK"),
    ]
    for i in range(len(df)):
        if random.random() < 0.10:
            txt = df.at[i, "text"]
            if not isinstance(txt, str):
                continue
            a, b = random.choice(swaps)
            if a in txt:
                df.at[i, "text"] = txt.replace(a, b, 1)
                modified[i] = True

    return df, float(modified.mean())

# =============================================================================
# Derived SBT episodes
# =============================================================================
def extract_sbt_episodes(vitals_df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for icu_stay_id, g in vitals_df.sort_values(["icu_stay_id", "chart_time"]).groupby("icu_stay_id"):
        in_sbt = False
        start_time = None
        last_time = None

        for _, row in g.iterrows():
            t = row["chart_time"]
            is_sbt = (row.get("mode") == "DIRECT_OXYGEN")

            if is_sbt and not in_sbt:
                in_sbt = True
                start_time = t
                last_time = t
            elif is_sbt and in_sbt:
                last_time = t
            elif (not is_sbt) and in_sbt:
                end_time = last_time + pd.Timedelta(hours=1)
                duration_h = (end_time - start_time).total_seconds() / 3600.0
                out.append({
                    "icu_stay_id": icu_stay_id,
                    "sbt_start_time": start_time,
                    "sbt_end_time": end_time,
                    "sbt_duration_hours": duration_h,
                })
                in_sbt = False
                start_time = None
                last_time = None

        if in_sbt:
            end_time = last_time + pd.Timedelta(hours=1)
            duration_h = (end_time - start_time).total_seconds() / 3600.0
            out.append({
                "icu_stay_id": icu_stay_id,
                "sbt_start_time": start_time,
                "sbt_end_time": end_time,
                "sbt_duration_hours": duration_h,
            })

    return pd.DataFrame(out)

# =============================================================================
# Reporting
# =============================================================================
def report_missing_rates(vitals_df: pd.DataFrame, mv_df: pd.DataFrame, lab_df: pd.DataFrame,
                         fluids_df: pd.DataFrame, notes_modified_pct: float) -> None:
    print("\n=== Reporting ===")

    print("\nVitals missing rate per column:")
    for col in _VITALS_NUM_COLS:
        if col in vitals_df.columns:
            print(f"  {col}: {float(vitals_df[col].isna().mean()):.3%}")
    for col in ["resp_device", "mode", "fio2_equiv_pct"]:
        if col in vitals_df.columns:
            print(f"  [must-not-miss] {col}: {float(vitals_df[col].isna().mean()):.3%}")

    print("\nMV missing rate per vent column:")
    for col in _MV_VENT_COLS:
        if col in mv_df.columns:
            print(f"  {col}: {float(mv_df[col].isna().mean()):.3%}")

    if "value" in lab_df.columns:
        print(f"\nLabs missing rate (value NaN): {float(lab_df['value'].isna().mean()):.3%}")

    if {"intake_ml", "output_ml"}.issubset(fluids_df.columns):
        days_missing = (fluids_df["intake_ml"].isna() | fluids_df["output_ml"].isna()).mean()
        print(f"\nFluids: % days missing intake or output: {float(days_missing):.3%}")

    print(f"\nNotes: % modified: {notes_modified_pct:.3%}")

# =============================================================================
# MAIN
# =============================================================================
def main():
    np.random.seed(SEED)
    random.seed(SEED)

    patients = []
    stays = []
    vitals_clean_all = []
    mv_all = []
    lab_all = []
    micro_all = []
    fluids_all = []
    notes_all = []
    med_orders_all = []
    med_admin_all = []

    # Create EXACT patient site assignments per SITE_COUNTS
    patient_site_list = []
    for site, count in SITE_COUNTS.items():
        patient_site_list += [site] * count
    random.shuffle(patient_site_list)

    patient_ids = [uid("pt") for _ in range(N_PATIENTS)]

    for patient_id, site in zip(patient_ids, patient_site_list):
        patients.append(generate_patient(patient_id=patient_id, site=site))

        # number of ICU stays for this patient: "some but not all"
        # 1 stay: 88%, 2 stays: 10%, 3 stays: 2%
        r = random.random()
        if r < 0.02:
            n_stays_for_patient = 3
        elif r < 0.12:
            n_stays_for_patient = 2
        else:
            n_stays_for_patient = 1

        # first stay base admit distributed across first 18 months to allow readmissions
        base_admit = START_DATE + pd.Timedelta(days=random.randint(0, int(365 * 1.5)))

        prev_discharge = None

        for stay_i in range(n_stays_for_patient):
            if stay_i == 0:
                admit_rough = base_admit
            else:
                # enforce "some within 6 months" in this block
                if random.random() < 0.65:
                    gap_days = random.randint(7, 180)
                else:
                    gap_days = random.randint(7, 365)
                admit_rough = prev_discharge + pd.Timedelta(days=gap_days)

            if admit_rough >= END_DATE:
                break

            admit, discharge = generate_stay_times(admit_rough)
            if discharge > END_DATE:
                discharge = END_DATE

            truth = generate_truth_course(admit, discharge)

            # Use patient's site for all stays (no cross-hospital transfers)
            icu_unit = _black_lion_unit_choice() if site == "BLACK_LION" else None
            assigned_machine_bed = _assigned_machine_bed(site, icu_unit)
            mv_data_available = 0 if site == "YEKATIT" else 1

            # If not a machine bed, bias toward never-intubated
            if assigned_machine_bed == 0 and (not truth.scenario.startswith("never_intubated")):
                if random.random() < 0.50:
                    truth = StayTruth(
                        scenario=weighted_choice([("never_intubated_o2", 0.75), ("never_intubated_room_air", 0.25)]),
                        admit=truth.admit, discharge=truth.discharge,
                        extub_time=None, reintub_time=None, death_time=truth.death_time,
                    )

            icu_stay_id = uid("stay")
            stay_vars = generate_static_stay_vars(truth)

            # Readmission flag within 6 months
            if prev_discharge is None:
                readmit_within_6m = 0
                icu_admission_number = 1
            else:
                readmit_within_6m = int(0 <= (truth.admit - prev_discharge).days <= 180)
                icu_admission_number = stay_i + 1

            stays.append({
                "icu_stay_id": icu_stay_id,
                "patient_id": patient_id,
                "site": site,
                "icu_unit": icu_unit,
                "assigned_machine_bed": assigned_machine_bed,
                "mv_data_available": mv_data_available,
                "icu_admission_number": icu_admission_number,
                "readmission_within_6m": readmit_within_6m,
                **stay_vars,
            })

            shock_any = int(stay_vars["shock_any"])
            vap_any = int(stay_vars["vap_any"])
            aki_any = int(stay_vars["aki_any"])

            vitals_clean = generate_vitals_hourly_with_resp_delivery(icu_stay_id, truth, shock_any=shock_any)
            vitals_clean_all.append(vitals_clean)

            mv_clean = generate_mv_hourly_from_vitals(icu_stay_id, truth, vitals_clean, mv_data_available=mv_data_available)
            mv_all.append(mv_clean)

            lab_all.append(generate_routine_lab_events(icu_stay_id, truth, shock_any=shock_any, vap_any=vap_any))
            micro_all.append(generate_micro_orders(icu_stay_id, truth, shock_any=shock_any, vap_any=vap_any))

            fluids_df = generate_fluid_balance_daily(icu_stay_id, truth, aki_any=aki_any, shock_any=shock_any)
            fluids_all.append(fluids_df)

            med_orders, med_admin = generate_med_orders_and_admin(icu_stay_id, truth, stay_vars, vitals_clean)
            med_orders_all.append(med_orders)
            med_admin_all.append(med_admin)

            notes_all.append(generate_notes(icu_stay_id, truth, vitals_clean, mv_clean, fluids_df, med_orders, med_admin))

            prev_discharge = truth.discharge

    patients_df = pd.DataFrame(patients)
    stays_df = pd.DataFrame(stays)

    vitals_clean_df = pd.concat(vitals_clean_all, ignore_index=True) if vitals_clean_all else pd.DataFrame()
    mv_df = pd.concat(mv_all, ignore_index=True) if mv_all else pd.DataFrame()
    lab_df = pd.concat(lab_all, ignore_index=True) if lab_all else pd.DataFrame()
    micro_df = pd.concat(micro_all, ignore_index=True) if micro_all else pd.DataFrame()
    fluids_df = pd.concat(fluids_all, ignore_index=True) if fluids_all else pd.DataFrame()
    notes_df = pd.concat(notes_all, ignore_index=True) if notes_all else pd.DataFrame()
    med_orders_df = pd.concat(med_orders_all, ignore_index=True) if med_orders_all else pd.DataFrame()
    med_admin_df = pd.concat(med_admin_all, ignore_index=True) if med_admin_all else pd.DataFrame()

    # Apply noise
    vitals_df = apply_vitals_noise(vitals_clean_df, stays_df=stays_df) if len(vitals_clean_df) else vitals_clean_df
    mv_df = apply_mv_noise(mv_df, vitals_df=vitals_clean_df) if len(mv_df) else mv_df
    lab_df = apply_lab_noise(lab_df, stays_df=stays_df) if len(lab_df) else lab_df
    fluids_df = apply_fluids_noise(fluids_df) if len(fluids_df) else fluids_df
    micro_df = apply_micro_noise(micro_df) if len(micro_df) else micro_df
    notes_df, notes_modified_pct = apply_notes_noise(notes_df) if len(notes_df) else (notes_df, 0.0)

    sbt_df = extract_sbt_episodes(vitals_df) if len(vitals_df) else pd.DataFrame()

    # Export
    patients_df.to_csv("patients.csv", index=False)
    stays_df.to_csv("icu_stays.csv", index=False)
    vitals_df.to_csv("vitals_hourly.csv", index=False)
    mv_df.to_csv("mv_hourly.csv", index=False)
    lab_df.to_csv("lab_events.csv", index=False)
    micro_df.to_csv("micro_orders.csv", index=False)
    fluids_df.to_csv("fluid_balance_daily.csv", index=False)
    notes_df.to_csv("notes.csv", index=False)
    med_orders_df.to_csv("med_orders.csv", index=False)
    med_admin_df.to_csv("med_admin.csv", index=False)
    sbt_df.to_csv("sbt_episodes.csv", index=False)

    print(f"Generated {len(patients_df)} patients and {len(stays_df)} ICU stays")
    print("Wrote: patients.csv, icu_stays.csv, vitals_hourly.csv, mv_hourly.csv, "
          "lab_events.csv, micro_orders.csv, fluid_balance_daily.csv, notes.csv, "
          "med_orders.csv, med_admin.csv, sbt_episodes.csv")

    report_missing_rates(vitals_df, mv_df, lab_df, fluids_df, notes_modified_pct)

if __name__ == "__main__":
    main()

/tmp/ipython-input-1421/3336015932.py:1723: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  vitals_clean_df = pd.concat(vitals_clean_all, ignore_index=True) if vitals_clean_all else pd.DataFrame()
/tmp/ipython-input-1421/3336015932.py:1724: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  mv_df = pd.concat(mv_all, ignore_index=True) if mv_all else pd.DataFrame()
/tmp/ipython-input-1421/3336015932.py:1727: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecate

Generated 6039 patients and 6851 ICU stays
Wrote: patients.csv, icu_stays.csv, vitals_hourly.csv, mv_hourly.csv, lab_events.csv, micro_orders.csv, fluid_balance_daily.csv, notes.csv, med_orders.csv, med_admin.csv, sbt_episodes.csv

=== Reporting ===

Vitals missing rate per column:
  hr_bpm: 0.840%
  rr_bpm: 0.950%
  spo2_pct: 0.729%
  map_mmhg: 4.082%
  sbp_mmhg: 0.807%
  dbp_mmhg: 0.758%
  temp_c: 1.279%
  glucose_mgdl: 1.243%
  o2_flow_lpm: 69.269%
  [must-not-miss] resp_device: 0.000%
  [must-not-miss] mode: 0.000%
  [must-not-miss] fio2_equiv_pct: 0.000%

MV missing rate per vent column:
  peep: 47.240%
  pip_cmH2O: 47.240%
  tv_set_ml: 62.531%
  rr_set: 47.240%

Labs missing rate (value NaN): 6.335%

Fluids: % days missing intake or output: 10.074%

Notes: % modified: 26.950%
